# Neutron Response Matrix loading test

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
dir = Path() / "response_matrix"

In [ ]:
output_list = [x for x in dir.iterdir() if "output" in x.stem]
output_list

In [ ]:
testfile = dir / "output_1MeV.txt"

In [ ]:
widths = [15, 15, 20, 15, 15, 15, 15, 15]
df = pd.read_fwf(testfile, widths=widths)
df

In [ ]:
df["source_e (MeV)"][0]

In [ ]:
df["det_pulse (MeVee)"].max()

In [ ]:
cut = pd.cut(df["det_pulse (MeVee)"], np.arange(0, 7, 0.005))
cut_index = cut.cat.categories

In [ ]:
df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0)

In [ ]:
new_df = pd.DataFrame(df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0))

In [ ]:
new_df

In [ ]:
old_index = new_df.index
mids = new_df.index.mid.to_series(index=cut_index)
new_df["mids"] = mids
new_df

In [ ]:
new_df['NPS'].to_numpy()

In [ ]:
source_e = df["source_e (MeV)"][0]
all(df["source_e (MeV)"] == source_e)

In [ ]:
response_sims = []
response_Ls = None
widths = [15, 15, 20, 15, 15, 15, 15, 15]

for output_file in output_list:
    df = pd.read_fwf(output_file, widths=widths)
    source_e = df["source_e (MeV)"][0].astype(np.float64)
    assert all(df["source_e (MeV)"] == source_e)
    cut = pd.cut(df["det_pulse (MeVee)"], np.arange(0, 7.005, 0.005))
    cut_index = cut.cat.categories
    new_df = pd.DataFrame(df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0))
    old_index = new_df.index
    mids = new_df.index.mid.to_series(index=cut_index)
    print(new_df.index)
    print(type(new_df.index))
    np_cps = new_df["NPS"].to_numpy()
    np_Ls = mids.to_numpy()
    response_sims.append((source_e, np_cps))
    if response_Ls is None:
        response_Ls = np_Ls
    else:
        assert response_Ls.shape == np_Ls.shape
        assert all(response_Ls == np_Ls)

response_sims = sorted(response_sims, key=lambda x: x[0])

In [ ]:
response_sims

In [ ]:
response_Ls

In [ ]:
for E, sim in response_sims:
    print(f"{E}: {sim.shape}")
print(response_Ls.shape)

In [ ]:
sim_Es, sim_histos = zip(*response_sims)
sim_Es

In [ ]:
np_R = np.array(sim_histos).T

In [ ]:
np_R.shape

In [ ]:
np_Es = np.array(sim_Es)
np_Es.shape